Prendete un grafo di 4-5 nodi, con almeno un nodo senza archi uscenti (dangling) e un ciclo.

In [21]:
import numpy as np

# Directed edge (i, j): a link from page i to page j. (i-->j) 
# Direction matters: (0, 2) and (2, 0) are two different edges.
#  edges i -> j (nodes 0..4); node 4 has no outgoing links (dangling)

edges = [(0, 1), (0, 2), (1, 2), (2, 0), (3, 2), (3, 4)]

n = 5                        # number of nodes

A = np.zeros((n,n))          ## Adiancy matrix nxn --> it describes the graph. --> it says if the link exist or not
print(A)

for i, j in edges:
    A[j, i] = 1              # column i holds the out-links of node i
print("\n Position of the edges to A: \n",A)

out = A.sum(axis=0)          # Out-degree of node i: number of edges leaving i (links on page i).
                             # For the example graph: out = [2, 1, 1, 2, 0]  

                             # the random surfer picks each outgoing link with equal probability.
                             # A NODE with OUT-DEGREE 0 is a DANGLING NODE --> its column stays all zeros
                             # and is later replaced by the teleportation vector v.
 
# Boolean mask marking dangling nodes (nodes with no outgoing links, like the last column, the n°4)
mk_dangling = (out == 0) # [False, False, False, False, True]

# Creation of Transition matrix P: is built by dividing each column of A by its out-degree so it sums to 1.
## Dangling columns remains all zeros (division is skipped where out == 0). 

## np.divide(A, out) divide A for the vect out. out it's allined to the columns of A, so column i of A is divided for out[i]
# where=out > 0 says that it performs the division only where=out > 0 so 1,2 ecc. not 0
# out=np.zeros_like(A) it's the array where the result division is written
P = np.divide(A, out, out=np.zeros_like(A), where=out > 0)

print("\n Transition matrix P: \n",P)

# Damping factor: probability of following a link (1 - alpha = teleportation that means the ignore the link and jumps to another page).
# It guarantees a unique positive PageRank vector and controls both the convergence rate (|lambda_2| <= alpha) and the sensitivity (~ 1/(1 - alpha)).
alpha = 0.85

# Teleportation vector: uniform over the n nodes. It's a distribution prob that a surfer choose the arrive page.
# n components that sum up to 1. It's uniform so 1/n for each page is equally probable.
# with another v we create a personalized PageRank
v = np.ones(n) / n
print("\n Teleportation vect v:",v)

# We want P column-stochastic matrix
# np.outer(v, dangling) built a 5x5 matrix where the i-column is equal to v if the i-node id dangling  othewise is all zero
P_fixed = P + np.outer(v, mk_dangling)

print("\n Fixed_Transition matrix P (it sums up to 1): \n",P_fixed)


# Now i've to build the Google Matrix --> with probability alpha the surfer follows the links of P_fixed,
# with probability 1 - alpha it teleports according to v. # np.outer(v, np.ones(n)) is the RANK-ONE matrix v*e^T (every column equals v).
# v e^T = [ v  v  v  v  v ]    (5 columns, every one is v) ; v is column vect nx1 and e^T is 1xn
teleport_matrix = np.outer(v, np.ones(n))
print("\n v*e^T",teleport_matrix)
G = alpha * P_fixed + (1 - alpha) * teleport_matrix

print("\nGoogle Matrix:\n",G)

##  G must be column-stochastic (each column sums to 1).
print("\nColumn sums of G:", G.sum(axis=0))
assert np.allclose(G.sum(axis=0), 1)


# Reference PageRank (dense eigen-decomposition, only for small tests):
# eigenvector of G for the eigenvalue 1, made positive and normalized to sum 1.
w, V = np.linalg.eig(G)        # calculate eigenvalues
k = np.argmax(w.real)          # index of the dominant eigenvalue
x_ref = np.abs(V[:, k].real)   # eigenvectors are defined up to sign
x_ref /= x_ref.sum()           # normalize so the entries sum to 1

print("\nDominant eigenvalue:", w.real[k]) # 0.9999999999999987
print("PageRank (reference):", x_ref) # [0.35017836 0.1884167  0.36539702 0.03959089 0.05641702]


[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

 Position of the edges to A: 
 [[0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 1. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]]

 Transition matrix P: 
 [[0.  0.  1.  0.  0. ]
 [0.5 0.  0.  0.  0. ]
 [0.5 1.  0.  0.5 0. ]
 [0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.5 0. ]]

 Teleportation vect v: [0.2 0.2 0.2 0.2 0.2]

 Fixed_Transition matrix P (it sums up to 1): 
 [[0.  0.  1.  0.  0.2]
 [0.5 0.  0.  0.  0.2]
 [0.5 1.  0.  0.5 0.2]
 [0.  0.  0.  0.  0.2]
 [0.  0.  0.  0.5 0.2]]

 v*e^T [[0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]
 [0.2 0.2 0.2 0.2 0.2]]

Google Matrix:
 [[0.03  0.03  0.88  0.03  0.2  ]
 [0.455 0.03  0.03  0.03  0.2  ]
 [0.455 0.88  0.03  0.455 0.2  ]
 [0.03  0.03  0.03  0.03  0.2  ]
 [0.03  0.03  0.03  0.455 0.2  ]]

Column sums of G: [1. 1. 1. 1. 1.]

Dominant eigenvalue: 0.9999999999999987
PageRank (reference): [0.35017836 0.1884167  0.365

In [22]:
import numpy as np
import scipy.sparse as sp


# Define a function that build the adjacency matrix --> it takes the list of edges of the graph and turn it into A
def build_adjacency(edges, n):

    # edges: list of directed edges (i, j), a link from page i to page j, and n: number of nodes
    # Returns the SPARSE adjacency matrix A with A[j, i] = 1 for each edge i -> j.

    # Step 1: keep each edge Only once and ignore self-loops [like (2,2) or (1,1) ]
    clean_edges = set()                 # a set() CANNOT contain the SAME ELEMENT twice
    for edge in edges:
        i = edge[0]                     # source: where the link starts --> i
        j = edge[1]                     # destination: where the link ends/arrives --> j
        if i != j:                      # skip self-loops (page linking to itself)
            clean_edges.add((i, j))     # adding an edge already present does nothing

    # Step 2: SPLIT the edges (without self-loops, ) into 2 lists, sources and destinations.
    src = []                            # SOURCE of each edge  -> column index of A
    dst = []                            # DESTINATION of each edge -> row index of A
    for edge in clean_edges:
        i = edge[0]
        j = edge[1]
        src.append(i)
        dst.append(j)

    # Step 3: one "1" for every edge. --> it creates a vector of 1 for every edges, could be every length
    data = np.ones(len(clean_edges))

    # Step 4: csr_matrix build the sparse matrix. Entry (row, column) = (dst, src), so column i holds the out-links of node i.
    # shape=(n, n) --> dimension; data are the values to put in the A matrix;  (dst, src) is a tuple that contains 2 lists
    A = sp.csr_matrix((data, (dst, src)), shape=(n, n))


    return A

In [36]:
# RANK SINK

#  Nodes 0, 1, 2 form the "main" part; nodes 3 and 4 form the sink.
# The sink only links to itself: 3 -> 4 and 4 -> 3. There is one way in: 2 -> 3.

edges_sink = [(0, 1), (1, 2), (2, 0), (2, 3), (3, 4), (4, 3)]
n_edges= 5
A_sink = build_adjacency(edges_sink, n_edges)     # is a sparse matrix

A = A_sink.toarray()  # Save A_sink in A ## is a dense copy only for the test --> to array coverts it in a dense matrix (simple array nxn)
print("\nA_sink or A:\n",A_sink.toarray())

out = A.sum(axis=0)
print("\nout:\n",out)

P = np.divide(A, out, out=np.zeros_like(A), where=out > 0)
print("\nP:\n",P)



A_sink or A:
 [[0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

out:
 [1. 1. 2. 1. 1.]

P:
 [[0.  0.  0.5 0.  0. ]
 [1.  0.  0.  0.  0. ]
 [0.  1.  0.  0.  0. ]
 [0.  0.  0.5 0.  1. ]
 [0.  0.  0.  1.  0. ]]


### <mark> Why convert a sparse matrix with .toarray()

A_sink is a sparse matrix (csr_matrix). A = A_sink.toarray() returns a dense NumPy copy (n × n, with all the zeros stored), used in tests on small graphs.

1. The experiment code uses dense-array operations

Lines such as np.zeros_like(A), np.divide(A, out, ...) and np.outer(...) are written for NumPy arrays. With a sparse matrix, some of these functions do not behave as you would expect: they may raise errors or give different results. Converting first avoids the problem.

2. To see the matrix

If you print a sparse matrix, Python only shows the list of occupied positions (row, column, value). With .toarray() you see it as a 5 × 5 table.

### <mark> Surfer WITHOUT teleportation<>

In [ ]:
# now we've to consider surfer WITHOUT teleportation that means (alpha=1) --> G = 1*P + 0 *veT
# start from the uniform distribution 1/n_edges

x = np.ones(n_edges) / n_edges
print("start:", x,"\n")

#Now apply P repeatedly. x[i] = probability that the surfer is on page i.
for k in range(1, 201):
    x = P @ x                      # one step of the random surfer
    if k in (1, 3, 10, 50, 199, 200):
        print(k, np.round(x, 4))

start: [0.2 0.2 0.2 0.2 0.2] 

1 [0.1 0.2 0.2 0.3 0.2]
3 [0.1 0.1 0.1 0.4 0.3]
10 [0.0125 0.025  0.025  0.4375 0.5   ]
50 [0.     0.     0.     0.4667 0.5333]
199 [0.     0.     0.     0.5333 0.4667]
200 [0.     0.     0.     0.4667 0.5333]


### <mark> Observation: Surfer without teleportation (rank sink graph)

1. **The probability leaves nodes 0, 1, 2.** It goes from 0.1-0.2 to 0.0125 at step 10 and to 0 at step 50. Every time the surfer goes through node 2, half of the probability takes the link to node 3 and enters the sink. It never leaves the sink, so the main part empties out.

2. **All the probability ends up in nodes 3 and 4.** At step 50 the values are 0.4667 and 0.5333, which still sum to 1: the probability does not disappear, it accumulates in the sink.

3. **But nodes 3 and 4 do not settle.** Compare rows 199 and 200: at 199 we have `0.5333 0.4667`, at 200 we have `0.4667 0.5333`. The values swap at every step, because the sink is a cycle of length 2 (3 → 4 → 3) and the probability bounces back and forth.

### <mark> Surfer with teleportation

In [ ]:
# now we've to consider surfer with teleportation, and we still consider uniform distribution 1/n_edges

alpha = 0.85
v = np.ones(n_edges) / n_edges          #teleportation vector
# build Google Matrix

v_eT = np.outer(v,np.ones(n_edges))
G = alpha*P + (1-alpha)*v_eT        # NO dangling nodes here, so P is already stochastic


## same iteration with G 

x = np.ones(5) / 5
for k in range(1, 201):
    x = G @ x
    if k in (1, 3, 10, 50, 199, 200):
        som = x.sum(axis=0).np.round(2)
        print(k, np.round(x, 4),som)

1 [0.115 0.2   0.2   0.285 0.2  ] 1.0000000000000002
3 [0.115  0.1278 0.1386 0.3464 0.2723] 1.0000000000000002
10 [0.0784 0.0988 0.1139 0.3613 0.3476] 1.0000000000000004
50 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0000000000000022
199 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0000000000000062
200 [0.0773 0.0957 0.1114 0.3706 0.345 ] 1.0000000000000062
